# Creating the Simplified Dataset

The current portal project uses R to create the new dataset. In order to understand the data pipeline, I'm recreating the process in python and documenting it here. 

#### Imports

In [30]:
import pandas as pd
import numpy as np
import os

### Full Portal Rodent Dataset

In [23]:
full_rodents = pd.read_csv("data/Portal_rodent.csv")
full_rodents.head()

C:\Users\12248\AppData\Local\Temp\ipykernel_23196\3720788422.py:1: DtypeWarning: Columns (21,22,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  full_rodents = pd.read_csv("data/Portal_rodent.csv")


,recordID,month,day,year,period,plot,note1,stake,species,sex,...,ltag,note3,prevrt,prevlet,nestdir,neststk,note4,note5,pit_tag,id
0,1,7,16,1977,1.0,2.0,NaN,16.0,NaN,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981505_NA_1
1,2,7,16,1977,1.0,3.0,NaN,23.0,NaN,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981506_NA_1
2,3,7,16,1977,1.0,2.0,NaN,25.0,DM,F,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981507_DM_1
3,4,7,16,1977,1.0,7.0,NaN,25.0,DM,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981508_DM_1
4,5,7,16,1977,1.0,3.0,NaN,26.0,DM,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981509_DM_1


In [24]:
full_rodents['note1'].value_counts()

note1
1.0     2095
13.0    1546
2.0     1297
5.0      850
3.0      850
9.0      442
6.0      411
4.0      350
8.0      117
10.0      63
11.0      54
16.0      14
12.0      13
14.0       4
17.0       4
15.0       1
Name: count, dtype: int64

### Standardize Full Dataset Columns

This step standardizes the full portal rodent table into a dchema that matches the teaching styled dataset.

The current full dataset has the columns:
- 'recordID'
- 'month'
- 'day'
- 'year'
- 'period'
- 'plot'
- 'note1'
- 'stake'
- 'species'
- 'sex'
- 'reprod'
- 'age'
- 'testes'
- 'vagina'
- 'pregnant'
- 'nipples'
- 'lactation'
- 'hfl'
- 'wgt'
- 'tag'
- 'note2'
- 'ltag'
- 'note3',
- 'prevrt'
- 'prevlet'
- 'nestdir'
- 'neststk'
- 'note4'
- 'note5'
- 'pit_tag'
- 'id'

This step will standardize the column names where needed, to be more clear. We are looking for:
- record_id
- plot_id
- species_id
- hindfoot_length
- weight


In [25]:
def standardize_portal_rodent_columns(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    rename_map = {
        "recordID": "record_id",
        "plot": "plot_id",
        "species": "species_id",
        "hfl": "hindfoot_length",
        "wgt": "weight",
    }

    existing_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_map)

    return df

standardized_full = standardize_portal_rodent_columns(full_rodents)
standardized_full.head()

,record_id,month,day,year,period,plot_id,note1,stake,species_id,sex,...,ltag,note3,prevrt,prevlet,nestdir,neststk,note4,note5,pit_tag,id
0,1,7,16,1977,1.0,2.0,NaN,16.0,NaN,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981505_NA_1
1,2,7,16,1977,1.0,3.0,NaN,23.0,NaN,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981506_NA_1
2,3,7,16,1977,1.0,2.0,NaN,25.0,DM,F,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981507_DM_1
3,4,7,16,1977,1.0,7.0,NaN,25.0,DM,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981508_DM_1
4,5,7,16,1977,1.0,3.0,NaN,26.0,DM,M,...,0,NaN,0,0,NaN,0.0,NaN,NaN,False,981509_DM_1


### Drop complex full data

The goal for this step is to get rid of data that was deemed uneccesary for the teaching dataset

In [26]:
def drop_full_dataset_complexity(df: pd.DataFrame) -> pd.DataFrame:
   
    df = df.copy()

    cols_to_drop = [
        "period",
        "stake",
        "reprod",
        "age",
        "testes",
        "vagina",
        "pregnant",
        "nipples",
        "lactation",
        "tag",
        "note2",
        "ltag",
        "note3",
        "prevrt",
        "prevlet",
        "nestdir",
        "neststk",
        "note4",
        "note5",
        "pit_tag",
        "id",
    ]

    existing = [c for c in cols_to_drop if c in df.columns]
    return df.drop(columns=existing)

simplified_full = drop_full_dataset_complexity(standardized_full)
simplified_full.head()

,record_id,month,day,year,plot_id,note1,species_id,sex,hindfoot_length,weight
0,1,7,16,1977,2.0,NaN,NaN,M,32.0,NaN
1,2,7,16,1977,3.0,NaN,NaN,M,33.0,NaN
2,3,7,16,1977,2.0,NaN,DM,F,37.0,NaN
3,4,7,16,1977,7.0,NaN,DM,M,36.0,NaN
4,5,7,16,1977,3.0,NaN,DM,M,35.0,NaN


### Optional Data Cleaning

Apply functions to make the data more teaching like, mostly including of removing missing data

In [27]:

def apply_teaching_simplifications(
    df: pd.DataFrame,
    drop_missing_species: bool = True,
    drop_missing_weight: bool = False,
    drop_missing_hindfoot: bool = False,
    valid_sex_only: bool = False
) -> pd.DataFrame:
    
    df = df.copy()

    if drop_missing_species and "species_id" in df.columns:
        df = df.dropna(subset=["species_id"])

    if drop_missing_weight and "weight" in df.columns:
        df = df.dropna(subset=["weight"])

    if drop_missing_hindfoot and "hindfoot_length" in df.columns:
        df = df.dropna(subset=["hindfoot_length"])

    if valid_sex_only and "sex" in df.columns:
        df = df[df["sex"].isin(["M", "F"])]

    return df.reset_index(drop=True)

print(f'{simplified_full.shape} before teaching simplifications')
simplified_full = apply_teaching_simplifications(simplified_full, drop_missing_weight=True, drop_missing_hindfoot=True, valid_sex_only=True)
print(f'{simplified_full.shape} after teaching simplifications')

(79213, 10) before teaching simplifications
(70917, 10) after teaching simplifications


### Add Plot and Species Metadata

Two seperate functions which join in the metadata to the main cleaned dataset

In [28]:
def add_species_metadata(
    surveys_df: pd.DataFrame,
    species_df: pd.DataFrame
) -> pd.DataFrame:
    
    surveys_df = surveys_df.copy()
    species_df = species_df.copy()

    species_df = species_df.rename(columns={
        "species": "species_name"
    })

    return surveys_df.merge(species_df, on="species_id", how="left")


def add_plot_metadata(
    surveys_df: pd.DataFrame,
    plots_df: pd.DataFrame
) -> pd.DataFrame:
    
    surveys_df = surveys_df.copy()
    plots_df = plots_df.copy()

    return surveys_df.merge(plots_df, on="plot_id", how="left")


species = pd.read_csv("data/species.csv")
plots = pd.read_csv("data/plots.csv")
merged_full = add_species_metadata(simplified_full, species)
merged_full = add_plot_metadata(merged_full, plots)
print(f'{merged_full.shape} after adding metadata')
merged_full.head()

(70917, 14) after adding metadata


,record_id,month,day,year,plot_id,note1,species_id,sex,hindfoot_length,weight,genus,species_name,taxa,plot_type
0,63,8,19,1977,3.0,NaN,DM,M,35.0,40.0,Dipodomys,merriami,Rodent,Long-term Krat Exclosure
1,64,8,19,1977,7.0,NaN,DM,M,37.0,48.0,Dipodomys,merriami,Rodent,Rodent Exclosure
2,65,8,19,1977,4.0,NaN,DM,F,34.0,29.0,Dipodomys,merriami,Rodent,Control
3,66,8,19,1977,4.0,NaN,DM,F,35.0,46.0,Dipodomys,merriami,Rodent,Control
4,67,8,19,1977,7.0,NaN,DM,M,35.0,36.0,Dipodomys,merriami,Rodent,Rodent Exclosure


### Add Note Meanings

Function which join the note meanings from a seperate dataset

In [29]:
rodent_notes = pd.read_csv("data/Portal_rodent_datanotes.csv")

note_map = rodent_notes.set_index("note1")["meaning"]
merged_full["note1_meaning"] = merged_full["note1"].map(note_map)
merged_full[merged_full["note1"].notna()]

,record_id,month,day,year,plot_id,note1,species_id,sex,hindfoot_length,weight,genus,species_name,taxa,plot_type,note1_meaning
332,633,2,19,1978,8.0,1.0,DS,M,55.0,154.0,Dipodomys,spectabilis,Rodent,Control,"some missing data (i.e. stake missing, mass, s..."
345,648,2,20,1978,16.0,1.0,DM,F,37.0,46.0,Dipodomys,merriami,Rodent,Rodent Exclosure,"some missing data (i.e. stake missing, mass, s..."
346,649,2,20,1978,16.0,1.0,DM,M,35.0,48.0,Dipodomys,merriami,Rodent,Rodent Exclosure,"some missing data (i.e. stake missing, mass, s..."
395,729,3,13,1978,15.0,1.0,DM,M,36.0,46.0,Dipodomys,merriami,Rodent,Long-term Krat Exclosure,"some missing data (i.e. stake missing, mass, s..."
396,731,4,8,1978,2.0,9.0,DS,F,47.0,110.0,Dipodomys,spectabilis,Rodent,Control,outside plot on exterior grid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70898,79296,12,14,2025,3.0,1.0,PP,F,19.0,12.0,Chaetodipus,penicillatus,Rodent,Long-term Krat Exclosure,"some missing data (i.e. stake missing, mass, s..."
70901,79299,12,14,2025,5.0,1.0,DM,M,36.0,44.0,Dipodomys,merriami,Rodent,Rodent Exclosure,"some missing data (i.e. stake missing, mass, s..."
70904,79304,12,14,2025,8.0,1.0,PP,M,22.0,15.0,Chaetodipus,penicillatus,Rodent,Control,"some missing data (i.e. stake missing, mass, s..."
70905,79305,12,14,2025,7.0,1.0,PP,F,20.0,13.0,Chaetodipus,penicillatus,Rodent,Rodent Exclosure,"some missing data (i.e. stake missing, mass, s..."


### Save New CSV

Save merged CSV to 'ouput' folder

In [ ]:
# Use pathlib instead


os.makedirs("output", exist_ok=True)
merged_full.to_csv("output/merged_full.csv", index=False)